# OpenTracy — Context Layer Workflow

End-to-end test of the context layer ([ADR-0001](../docs/adr/0001-context-layer.md)): the ordered document stack the model sees every turn.

```
1. soul.md                    behavioral authority (hand-edited)
2. tools/descriptions.md      tools index (generated)
3. skills/descriptions.md     skills index (generated)
4. memory/user.md             who the user is (auto-updated)
5. memory/memory.md           working memory (auto-updated)
6. sessions/past_sessions.md  prior-session summaries (auto-updated)
7. <live messages>            owned by the session loop
```

**What this notebook exercises:**
1. Assembling the real workspace context — ordering, token accounting
2. A sandboxed **two-session memory lifecycle** — facts written in session 1 are recalled in session 2
3. Budgets & truncation — context stays O(budget), never O(history)
4. The optional **LangChain edge adapter**

In [1]:
import sys
from pathlib import Path

# Works installed (pip install -e .) or straight from a checkout.
ROOT = Path.cwd() if (Path.cwd() / "soul.md").exists() else Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from opentracy.core import ContextLayer
from opentracy.core.context import DEFAULT_STACK

print("workspace:", ROOT)
print("stack    :", [name for name, *_ in DEFAULT_STACK] + ["<live messages>"])

workspace: /Users/diogovieira/Developer/opentracy-build
stack    : ['soul', 'tools', 'skills', 'user', 'memory', 'past_sessions', '<live messages>']


## 1 · Assemble the real workspace context

One call loads every document, strips frontmatter, enforces budgets, and preserves the static→dynamic order (the prompt-cache contract).

In [2]:
ctx = ContextLayer.from_workspace(ROOT).assemble()

expected = [name for name, *_ in DEFAULT_STACK]
loaded = [b.source for b in ctx.blocks]
assert loaded == [n for n in expected if n in loaded], "stack order violated"

WIDTH, scale = 40, max(b.tokens for b in ctx.blocks)
for b in ctx.blocks:
    bar = "\u2588" * max(1, round(b.tokens / scale * WIDTH))
    flag = "  (truncated)" if b.truncated else ""
    print(f"{b.source:>14} \u2502 {bar:<{WIDTH}} {b.tokens:>5} tok{flag}")
print(f"{'TOTAL':>14} \u2502 {'':<{WIDTH}} {ctx.total_tokens:>5} tok")

          soul │ ████████████████████████████████████████   187 tok
         tools │ ██████████████████████                     102 tok
        skills │ ████████████████████                        93 tok
          user │ ███████████████████                         89 tok
        memory │ ███████████████████████████████████        164 tok
 past_sessions │ ████████████████████████                   113 tok
         TOTAL │                                            748 tok


In [3]:
prompt = ctx.system_prompt
assert "managed:" not in prompt, "frontmatter must never spend model tokens"
print(prompt[:400])
print("\n[\u2026 rendered blocks omitted \u2026]\n")
print(prompt[-300:])

<context source="soul">
# Soul

The highest-authority behavioral document in the context stack. Defines the user's
personality, preferences, tone, and behavioral profile — who the agent is working
*for* and how it must adapt. When any other context document conflicts with this
one, this one wins.

## Identity
<!-- Who the user is at the highest level, in one or two sentences. -->
-

## Tone & comm

[… rendered blocks omitted …]



## YYYY-MM-DD · <session-id> — <one-line outcome>
- **Goal:** what the user asked for
- **Outcome:** what actually happened (faithful — including failures)
- **Decisions:** choices made that future sessions must respect
- **Open threads:** unfinished work a future session may resume
-->
</context>


## 2 · Two-session memory lifecycle (sandboxed)

A throwaway workspace simulates the full loop. The write-back below is done by hand here — it is exactly what the **memory foundation (Phase 4)** will do automatically at session end, under its write policies (dedupe, update-over-duplicate, delete falsified).

In [4]:
import tempfile, textwrap

tmp = tempfile.TemporaryDirectory()
WS = Path(tmp.name)

def write(rel: str, body: str) -> None:
    path = WS / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(body).lstrip(), encoding="utf-8")

write("soul.md", """
    ---
    managed: human
    ---
    # Soul
    ## Tone & communication
    - Language: PT-BR for chat, EN for code
    - Verbosity: concise, no filler
    ## Behavioral profile
    - High autonomy; ask only before destructive actions
""")
write("skills/descriptions.md", """
    ---
    managed: runtime
    ---
    # Skills index
    | Skill | Pack | Use when |
    |---|---|---|
    | match-correction-triage | sharpi | A batch of match corrections needs triage |
""")
print("sandbox workspace:", WS)

sandbox workspace: /var/folders/2l/zd8kml2j349447847pkg4c3c0000gn/T/tmpxr5a8rz7


In [5]:
# ---- SESSION 1: fresh user, no memory yet ----
session1 = ContextLayer.from_workspace(WS).assemble()
print("session 1 blocks:", [b.source for b in session1.blocks])
assert [b.source for b in session1.blocks] == ["soul", "skills"]

# ---- session end: the runtime writes back what it learned ----
write("memory/user.md", """
    ---
    managed: runtime
    ---
    # User
    ## Profile
    - The user is a software engineer.
    ## Extracted facts
    - Works on Sharpi match quality \u00b7 first observed 2026-07-04
""")
write("memory/memory.md", """
    ---
    managed: runtime
    ---
    # Memory
    ## Recurring workflows
    - Runs correction-triage batches on Mondays; expects a summary table.
""")
write("sessions/past_sessions.md", """
    ---
    managed: runtime
    ---
    # Past sessions
    ## 2026-07-04 \u00b7 s-001 \u2014 triaged 42 corrections
    - **Goal:** triage the weekly correction batch
    - **Outcome:** 42 routed to eval datasets, 3 flagged for review
    - **Open threads:** 3 flagged corrections await user decision
""")
print("write-back complete \u2713")

session 1 blocks: ['soul', 'skills']
write-back complete ✓


In [6]:
# ---- SESSION 2: a new session recalls everything ----
session2 = ContextLayer.from_workspace(WS).assemble()
print("session 2 blocks:", [b.source for b in session2.blocks])
assert [b.source for b in session2.blocks] == [
    "soul", "skills", "user", "memory", "past_sessions"
]

prompt2 = session2.system_prompt
assert "software engineer" in prompt2
assert "3 flagged corrections await user decision" in prompt2
assert prompt2.index("# Soul") < prompt2.index("# User") < prompt2.index("# Past sessions")

print(f"\n\u2713 facts written in session 1 are recalled in session 2")
print(f"\u2713 context grew {session1.total_tokens} \u2192 {session2.total_tokens} tokens, order preserved")

session 2 blocks: ['soul', 'skills', 'user', 'memory', 'past_sessions']

✓ facts written in session 1 are recalled in session 2
✓ context grew 80 → 198 tokens, order preserved


## 3 · Budgets & truncation

Simulate months of accumulated facts, then assemble with a tight budget: the block truncates with a visible marker and a `truncated` flag in the report. In production this flag is the **compaction trigger** — cold facts shard to `memory/archive/` instead of growing the always-loaded document.

In [7]:
facts = "\n".join(f"- fact {i}: something observed on day {i}" for i in range(400))
write("memory/memory.md", f"---\nmanaged: runtime\n---\n# Memory\n{facts}")

tight = ContextLayer.from_workspace(WS, budget_overrides={"memory": 150}).assemble()
mem = next(b for b in tight.blocks if b.source == "memory")
assert mem.truncated and mem.tokens <= 170  # budget + truncation marker

print("report:", tight.report()["memory"])
print("block ends with:", repr(mem.content[-60:]))
print(f"\n✓ 400 facts (~4,000 tokens) on disk, only {mem.tokens} tokens in context")

report: {'tokens': 160, 'truncated': True}
block ends with: 'n day 14\n- fact 15:\n\n[... truncated to 150 token budget ...]'

✓ 400 facts (~4,000 tokens) on disk, only 160 tokens in context


## 4 · LangChain edge adapter (optional)

The core stays dependency-free; this adapter converts an assembled context + live history ("messages happening now", layer 7) into LangChain messages at the boundary. Requires `pip install 'opentracy[langchain]'`.

In [8]:
try:
    from opentracy.providers.langchain_adapter import to_langchain_messages

    msgs = to_langchain_messages(session2, history=[
        {"role": "user", "content": "What is still pending from last session?"},
        {"role": "assistant", "content": "3 flagged corrections await your decision."},
        {"role": "user", "content": "Show them."},
    ])
    print([type(m).__name__ for m in msgs])
    assert type(msgs[0]).__name__ == "SystemMessage"
    assert msgs[0].content == session2.system_prompt
    print(f"\n\u2713 SystemMessage carries the full context stack ({len(msgs[0].content):,} chars)")
except ImportError:
    print("langchain-core not installed \u2014 skipping. Install with: pip install 'opentracy[langchain]'")

['SystemMessage', 'HumanMessage', 'AIMessage', 'HumanMessage']

✓ SystemMessage carries the full context stack (977 chars)


In [9]:
tmp.cleanup()
print("sandbox removed \u2014 all checks passed \u2713")

sandbox removed — all checks passed ✓


## Wrap-up

| Verified | How |
|---|---|
| Static→dynamic stack order | asserted in §1 and §2 |
| Frontmatter never spends tokens | asserted in §1 |
| Two-session recall (write → reload) | §2 sandbox lifecycle |
| Budget enforcement + truncation flag | §3 |
| LangChain kept at the edge | §4 |

What the runtime will automate next: the §2 write-back becomes the **memory foundation** (Phase 4), and the §3 `truncated` flag drives **compaction** to `memory/archive/`. The `descriptions.md` indexes get generators in Phases 2–3.